# IMDB Movie Reviews Dataset - Text Preprocessing

## Overview
This dataset contains **50,000 movie reviews** from IMDB (Internet Movie Database) designed for binary sentiment classification and natural language processing tasks. It provides a comprehensive collection of highly polar movie reviews that can be used for text analytics, sentiment analysis, and machine learning applications.

## Dataset Structure

The dataset is organized as a CSV file with the following columns:

| Column | Description |
|--------|-------------|
| `review` | The text of the movie review (may contain HTML tags like `<br />`) |
| `sentiment` | Binary classification - either `positive` or `negative` |

## Dataset Composition

- **Total Reviews**: 50,000
- **Training Set**: 25,000 reviews
- **Test Set**: 25,000 reviews
- **Sentiment Distribution**: Balanced dataset with equal numbers of positive and negative reviews

## Source Information

- **Author**: Lakshmipathi N
- **Original Source**: [Stanford AI Lab - Large Movie Review Dataset](http://ai.stanford.edu/~amaas/data/sentiment/)
- **Kaggle Dataset**: [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/data)

---

## Preprocessing Objectives

This notebook demonstrates essential text preprocessing techniques for natural language processing. Each technique will be implemented, analyzed, and compared to understand its impact on the dataset.

### a) Tokenization

**Definition**: Tokenization is the process of breaking down text into smaller units called tokens (words, sentences, or subwords).

**Techniques to Implement**:
- **Word Tokenization**: Splitting text into individual words based on spaces and punctuation
- **Sentence Tokenization**: Dividing text into separate sentences using punctuation markers
- **Subword Tokenization**: Breaking words into smaller meaningful units (useful for handling rare words and morphological variations)

**Comparison Goals**:
- Analyze how different tokenization strategies handle contractions, punctuation, and special characters
- Evaluate vocabulary size produced by each method
- Discuss advantages: Word tokenization is simple and interpretable; sentence tokenization preserves context boundaries; subword tokenization handles out-of-vocabulary words effectively

### b) Removing Stop Words

**Definition**: Stop words are common words (e.g., "the", "is", "and") that occur frequently but often carry little semantic meaning for text analysis.

**Implementation Goals**:
- Identify the most common stop words in the IMDB reviews dataset
- Remove standard English stop words using predefined lists
- Analyze the impact on vocabulary size before and after removal

**Analysis Points**:
- Measure reduction in vocabulary size and feature space
- Evaluate information retention - determine if removing stop words affects sentiment indicators
- Discuss trade-offs between dimensionality reduction and potential loss of contextual information (e.g., "not good" vs "good")

### c) Stemming and Lemmatization

**Definition**: Both techniques reduce words to their base or root form, but use different approaches.

**Stemming**: Rule-based approach that removes word suffixes to find the stem
- **Porter Stemmer**: Classic algorithm with simple suffix-stripping rules
- **Snowball Stemmer**: Improved version of Porter, supports multiple languages

**Lemmatization**: Dictionary-based approach that reduces words to their dictionary form (lemma) considering part of speech

**Comparison Goals**:
- Apply both Porter and Snowball stemmers to the reviews
- Apply lemmatization using morphological analysis
- Compare the resulting forms: stemmers may produce non-words (e.g., "running" → "run" vs "runn"), while lemmatization produces valid words

**Discussion Points**:
- **Stemming Advantages**: Faster computation, simpler implementation, effective for reducing vocabulary
- **Stemming Disadvantages**: May produce non-words, over-stemming (grouping unrelated words), under-stemming (not reducing related words)
- **Lemmatization Advantages**: Produces linguistically correct base forms, better semantic preservation
- **Lemmatization Disadvantages**: Slower (requires POS tagging), more complex, requires dictionary lookup
- **Dataset-specific considerations**: For sentiment analysis, evaluate which method better preserves emotional intensity and meaning

### d) Text Vectorization

**Definition**: Converting text into numerical representations that machine learning algorithms can process.

**Techniques to Implement**:

1. **Bag-of-Words (BoW)**:
   - Represents text as word frequency counts
   - Creates a vocabulary of all unique words
   - Each document becomes a vector of word counts

2. **TF-IDF (Term Frequency-Inverse Document Frequency)**:
   - Weights words based on their importance in a document relative to the entire corpus
   - Reduces the impact of common words across all documents
   - Highlights words that are distinctive to specific documents

3. **Word Embeddings (Word2Vec or similar)**:
   - Represents words as dense vectors in continuous space
   - Captures semantic relationships between words
   - Pre-trained or custom-trained on the dataset

**Comparison Goals**:
- **Dimensionality**: Compare the size of feature vectors (BoW/TF-IDF: vocabulary size vs Word2Vec: typically 100-300 dimensions)
- **Sparsity**: Evaluate sparsity levels (BoW/TF-IDF: highly sparse vs Word2Vec: dense)
- **Semantic Meaning**: Assess whether similar sentiments produce similar vectors
- **Computational Efficiency**: Memory usage and processing time for each method

**Analysis Points**:
- How do different vectorization methods affect downstream sentiment classification?
- Which representation best captures sentiment polarity?
- Trade-offs between interpretability and performance

In [ ]:
import pandas as pd
import re

# Tokenize
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Stop word
from collections import Counter
from nltk.corpus import stopwords

# Import thư viện cho Stemming và Lemmatization
from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag

In [ ]:
df = pd.read_csv("IMDB Dataset.csv")

In [ ]:
def clean_text(text):
    # Xóa các tag HTML
    text = re.sub(r"<.*?>", " ", text)
    # Xóa các khoảng trắng bị thừa
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()

## Text Cleaning and Duplicate Removal

### Purpose
Before performing any NLP analysis, we need to ensure our dataset is clean and free from redundant information.

### Steps:
1. **Text Cleaning**: Apply the `clean_text()` function to remove HTML tags, special characters, and normalize whitespace
2. **Duplicate Removal**: Remove duplicate reviews to prevent bias in our analysis and model training

### Why Remove Duplicates?
- Prevents data leakage between training and test sets
- Reduces overfitting by eliminating exact duplicates
- Ensures each review contributes unique information to the analysis

In [ ]:
print(f"Kích thước dataset ban đầu: {len(df)} reviews")
print(f"Số lượng data bị trùng: {df.duplicated(subset=['review']).sum()}")

# Áp dụng clean text
df['cleaned_review'] = df['review'].apply(clean_text)

# Xóa các dòng bị trùng
df_cleaned = df.drop_duplicates(subset=['review'], keep='first')

print(f"\nSố lượng đánh giá sau khi xóa: {len(df_cleaned)} reviews")
print(f"\nDataset: {df_cleaned.shape}")

Kích thước dataset ban đầu: 50000 reviews
Số lượng data bị trùng: 418

Số lượng đánh giá sau khi xóa: 49582 reviews

Dataset: (49582, 3)


In [ ]:
# So sánh giữa data gốc và data đã clean
for i in range(2):
    print(f"Ví dụ {i+1}:")
    print('='*80)
    print(f"Gốc: {df_cleaned.iloc[i]['review'][:200]}...")
    print(f"Làm sạch:  {df_cleaned.iloc[i]['cleaned_review'][:200]}...")

Ví dụ 1:
Gốc: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo...
Làm sạch:  one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me. the first thing that struck me about oz was i...
Ví dụ 2:
Gốc: A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece...
Làm sạch:  a wonderful little production. the filming technique is very unassuming- very old-time-bbc fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. the actors...


## A) Tokenization (Phân tách từ)

### 3 Phương pháp Tokenization

#### 1. **Word Tokenization (Phân tách từ)**

* **Phương pháp**: Tách văn bản thành các từ riêng lẻ (tokens) dựa trên khoảng trắng và dấu câu.
* **Ví dụ minh họa**:
* *Input*: "It's a great movie!"
* *Output*: `["It", "'s", "a", "great", "movie", "!"]`



#### 2. **Sentence Tokenization (Phân tách câu)**

* **Phương pháp**: Tách cả đoạn văn đánh giá thành danh sách các câu riêng lẻ dựa trên dấu chấm câu (. ! ?).
* **Ví dụ minh họa**:
* *Input*: "The acting was terrible. I will never watch it again."
* *Output*: `["The acting was terrible.", "I will never watch it again."]`



#### 3. **Subword Tokenization (Phân tách từ con)**

* **Phương pháp**: Phân tách một từ phức tạp thành các đơn vị nhỏ hơn (subwords) mang ý nghĩa. Cách này giúp máy hiểu được các từ ghép hoặc từ chưa từng gặp.
* **Ví dụ minh họa 1**:
* *Input*: "Unbelievable"
* *Output*: `["un", "believ", "able"]`


* **Ví dụ minh họa 2 (Từ ghép trong review)**:
* *Input*: "Superheroic"
* *Output*: `["super", "hero", "ic"]`

### 1️⃣ Word Tokenization (Phân tách từ)

In [ ]:
sample_review = df_cleaned.iloc[0]['cleaned_review']
# Word Tokenization - Tách thành các từ riêng lẻ
word_tokens = word_tokenize(sample_review)

print("KẾT QUẢ WORD TOKENIZATION:")
print(f"Số lượng tokens: {len(word_tokens)}")
print(f"\n30 tokens đầu tiên:")
print(word_tokens[:30])
print(f"\nVí dụ cụ thể về cách tách:")
demo_text = "I'm a boy"
demo_tokens = word_tokenize(demo_text.lower())
print(f"  '{demo_text}' → {demo_tokens}")

# Áp dụng cho toàn bộ dataset
print("\n Word Tokenization cho dataset")
df_cleaned['word_tokens'] = df_cleaned['cleaned_review'].apply(word_tokenize)

KẾT QUẢ WORD TOKENIZATION:
Số lượng tokens: 359

30 tokens đầu tiên:
['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'oz', 'episode', 'you', "'ll", 'be', 'hooked', '.', 'they', 'are', 'right', ',', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with']

Ví dụ cụ thể về cách tách:
  'I'm a boy' → ['i', "'m", 'a', 'boy']

 Word Tokenization cho dataset


/var/folders/zc/qmbjdwpx1zj85jxfxr58_vf40000gn/T/ipykernel_73910/672385649.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['word_tokens'] = df_cleaned['cleaned_review'].apply(word_tokenize)


### 2️⃣ Sentence Tokenization (Phân tách câu)

In [ ]:
# Sentence Tokenization - Tách thành các câu
sentence_tokens = sent_tokenize(sample_review)

print("KẾT QUẢ SENTENCE TOKENIZATION:")
print(f"Số lượng câu: {len(sentence_tokens)}")
print(f"\n3 câu đầu tiên:")
for i, sent in enumerate(sentence_tokens[:3], 1):
    print(f"{i}. {sent}")

# Áp dụng cho toàn bộ dataset
print("\n Sentence Tokenization cho dataset...")
df_2 = df_cleaned.copy()
df_2['sentence_tokens'] = df_2['cleaned_review'].apply(sent_tokenize)
df_2['num_sentences'] = df_2['sentence_tokens'].apply(len)

print(f"\n Thống kê số câu trong dataset:")
print(df_2['num_sentences'].describe())

KẾT QUẢ SENTENCE TOKENIZATION:
Số lượng câu: 13

3 câu đầu tiên:
1. one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked.
2. they are right, as this is exactly what happened with me.
3. the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go.

 Sentence Tokenization cho dataset...

 Thống kê số câu trong dataset:
count    49582.000000
mean        12.170122
std          8.780834
min          1.000000
25%          7.000000
50%         10.000000
75%         15.000000
max        282.000000
Name: num_sentences, dtype: float64


### 3️⃣ Subword Tokenization (Phân tách từ con) - Khuyến nghị cho Deep Learning

https://huggingface.co/learn/llm-course/chapter6/5

In [ ]:
# Subword Tokenization sử dụng BPE (Byte Pair Encoding)
# Khởi tạo tokenizer với BPE - [UNK] là token đại diện cho ký tự lạ không thể xử lý (Unknown)
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# Cấu hình trainer
trainer = BpeTrainer( # BERT 
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]"],
    vocab_size=30000,  # Kích thước từ vựng
    min_frequency=2    # Tần suất tối thiểu
)

# Huấn luyện trên một mẫu nhỏ của dataset 
training_texts = df_cleaned['cleaned_review'].head(15000).tolist()
tokenizer.train_from_iterator(training_texts, trainer=trainer)

print(f"Kích thước từ vựng: {tokenizer.get_vocab_size()}")




Kích thước từ vựng: 30000


In [ ]:
# Demo Subword Tokenization
output = tokenizer.encode(sample_review)
subword_tokens = output.tokens

print("KẾT QUẢ SUBWORD TOKENIZATION (BPE):")
print(f"Số lượng subword tokens: {len(subword_tokens)}")
print(f"\n30 tokens đầu tiên:")
print(subword_tokens[:30])

KẾT QUẢ SUBWORD TOKENIZATION (BPE):
Số lượng subword tokens: 372

30 tokens đầu tiên:
['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'oz', 'episode', 'you', "'", 'll', 'be', 'hooked', '.', 'they', 'are', 'right', ',', 'as', 'this', 'is', 'exactly', 'what', 'happened']


## ✅ QUYẾT ĐỊNH: Sử dụng Subword Tokenization (BPE) cho Project

### 🎯 Lý do lựa chọn Subword Tokenization

#### 1. **Xử lý vấn đề Out-Of-Vocabulary (OOV)**
- Dataset IMDB chứa nhiều từ sáng tạo, từ ghép, và từ hiếm
- **Ví dụ thực tế**: "cinematographically" → `["cinemato", "graph", "ically"]`
- Mô hình vẫn hiểu được từ mới thông qua các subword components đã học

#### 2. **Giảm kích thước từ vựng đáng kể**
- **Word Tokenization**: ~150,000+ từ unique
- **Subword Tokenization (BPE)**: 10,000 subwords
- **Giảm 93%** kích thước từ vựng → Tiết kiệm bộ nhớ và tăng tốc training

#### 3. **Bảo toàn ý nghĩa ngữ nghĩa**
- Các từ có cùng gốc sẽ share subwords chung
- **Ví dụ**: "unbelievable", "believe", "believer" đều chứa "believ"
- Giúp mô hình học được mối quan hệ giữa các từ

#### 4. **Tối ưu cho Sentiment Analysis**
- Xử lý tốt các từ mang cảm xúc mạnh nhưng hiếm gặp
- **Ví dụ**: "superheroic" → `["super", "heroic"]` (cả 2 đều mang nghĩa tích cực)

In [ ]:
# Áp dụng Subword Tokenization cho toàn bộ dataset
print("🔄 Đang áp dụng Subword Tokenization (BPE) cho toàn bộ dataset...")

# Hàm tokenize sử dụng BPE tokenizer
def subword_tokenize(text):
    return tokenizer.encode(text).tokens

# Tạo cột mới với subword tokens
df_cleaned['subword_tokens'] = df_cleaned['cleaned_review'].apply(subword_tokenize)

🔄 Đang áp dụng Subword Tokenization (BPE) cho toàn bộ dataset...
✅ Hoàn thành!

📊 So sánh số lượng tokens trung bình:
   Word Tokenization:    264.9 tokens/review
   Subword Tokenization: 279.4 tokens/review

💡 Subword tokenization tạo nhiều tokens hơn ~5.5%
   nhưng vocabulary size nhỏ hơn nhiều (10,000 vs 150,000+)


/var/folders/zc/qmbjdwpx1zj85jxfxr58_vf40000gn/T/ipykernel_73910/3786303751.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['subword_tokens'] = df_cleaned['cleaned_review'].apply(subword_tokenize)


## B) Removing Stop Words (Loại bỏ Stop Words)

### Định nghĩa
Stop words là các từ phổ biến xuất hiện thường xuyên trong văn bản nhưng thường mang ít ý nghĩa ngữ nghĩa cho phân tích văn bản (ví dụ: "the", "is", "and", "a").

### ⚠️ Lưu ý với Subword Tokenization
**Vì chúng ta đã chọn Subword Tokenization (BPE)**, cách xử lý stop words sẽ khác so với Word Tokenization:

#### Với Word Tokenization (cách truyền thống):
- Stop words là các từ hoàn chỉnh: "the", "is", "and"
- Ta có thể loại bỏ chúng trực tiếp

#### Với Subword Tokenization (approach của chúng ta):
- Các stop words đã được phân tách thành subwords
- **Không cần loại bỏ stop words một cách thủ công** vì:
  - Vocabulary đã rất nhỏ (10,000 subwords)
  - Mô hình Deep Learning tự học được stop words ít quan trọng
  - Việc loại bỏ subwords có thể phá vỡ cấu trúc từ

### 💡 Demo mục đích học tập
Phần dưới đây sẽ demo cách **identify và remove stop words với Word Tokenization** để hiểu về stop words. Tuy nhiên, trong pipeline chính, **chúng ta sẽ sử dụng subword_tokens trực tiếp** mà không cần loại bỏ stop words.

### Xác định Stop Words phổ biến trong Dataset

In [ ]:
# Đếm tần suất tất cả các từ trong dataset
all_words = []
for tokens in df_cleaned['subword_tokens']:
    all_words.extend(tokens)

# Lấy top 20 từ phổ biến nhất
word_counter = Counter(all_words)
top_20_words = word_counter.most_common(20)

print(f"\nTOP 20 TỪ XUẤT HIỆN NHIỀU NHẤT:")
print("=" * 70)
print(f"{'Từ':<15} {'Tần suất':>15} {'% trong tổng số từ':>20}")
print("=" * 70)

total_words = len(all_words)
for word, count in top_20_words:
    percentage = (count / total_words) * 100
    print(f"{word:<15} {count:>15,} {percentage:>19.2f}%")


TOP 20 TỪ XUẤT HIỆN NHIỀU NHẤT:
Từ                     Tần suất   % trong tổng số từ
the                     663,288                4.79%
.                       528,709                3.82%
,                       515,472                3.72%
a                       323,603                2.34%
and                     322,418                2.33%
of                      287,531                2.08%
to                      267,235                1.93%
'                       255,706                1.85%
is                      210,866                1.52%
it                      190,186                1.37%
in                      187,996                1.36%
i                       176,468                1.27%
this                    149,837                1.08%
that                    142,828                1.03%
s                       130,101                0.94%
-                       109,122                0.79%
"                       101,525                0.73%
was          

Đa phần các từ xuất hiện nhiều nhất là các stop word: 
`the`, `and`, `a`, `of`, `to`, `is`, `it`, `in`, `this`, `that`

### Safe Stop Word Removal - Giữ lại từ phủ định

In [ ]:
# Lấy danh sách stop words tiếng Anh
stop_words = set(stopwords.words('english'))

# Danh sách các từ phủ định cần giữ lại
negation_words = {
    'no', 'not', 'nor', 'never', 
    "don't", "doesn't", "didn't", "won't", "wouldn't", "shouldn't",
    "wasn't", "weren't", "isn't", "aren't", "ain't",
    "hasn't", "haven't", "hadn't",
    "can't", "cannot", "couldn't",
    "mustn't", "mightn't", "needn't"
}

# Tùy chỉnh: Loại bỏ các từ phủ định khỏi danh sách stop words
custom_stop_words = stop_words - negation_words

In [ ]:
# Hàm loại bỏ stop words
def remove_stopwords(tokens):
    return [word for word in tokens if word not in custom_stop_words]

# Áp dụng cho toàn bộ dataset
df_cleaned['tokens_no_stopwords'] = df_cleaned['subword_tokens'].apply(remove_stopwords)

/var/folders/zc/qmbjdwpx1zj85jxfxr58_vf40000gn/T/ipykernel_73910/395063877.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['tokens_no_stopwords'] = df_cleaned['subword_tokens'].apply(remove_stopwords)


In [ ]:
# Demo với câu ví dụ
demo_sentence = "The movie is not good, it is boring."
print("🎬 DEMO: Kiểm chứng giữ lại từ phủ định")
print("=" * 80)
print(f"Câu gốc: '{demo_sentence}'")

# Tokenize câu demo
demo_tokens = subword_tokenize(demo_sentence.lower())
print(f"\nSau khi tokenize: {demo_tokens}")

# Loại bỏ stop words
demo_no_stopwords = remove_stopwords(demo_tokens)
print(f"\nSau khi loại bỏ stop words: {demo_no_stopwords}")

print("=" * 80)

🎬 DEMO: Kiểm chứng giữ lại từ phủ định
Câu gốc: 'The movie is not good, it is boring.'

Sau khi tokenize: ['the', 'movie', 'is', 'not', 'good', ',', 'it', 'is', 'boring', '.']

Sau khi loại bỏ stop words: ['movie', 'not', 'good', ',', 'boring', '.']


In [ ]:
for i in range(2):
    print(f"\n🎬 Review {i+1}:")
    print("-" * 80)
    
    # Lấy 15 từ đầu tiên
    before = df_cleaned.iloc[i]['subword_tokens'][:15]
    after = df_cleaned.iloc[i]['tokens_no_stopwords'][:15]
    
    print(f"TRƯỚC (15 từ đầu): {before}")
    print(f"SAU   (15 từ đầu): {after}")
    
    # Tính tỷ lệ giảm
    len_before = len(df_cleaned.iloc[i]['subword_tokens'])
    len_after = len(df_cleaned.iloc[i]['tokens_no_stopwords'])
    reduction = ((len_before - len_after) / len_before) * 100
    
    print(f"Giảm: {len_before} → {len_after} tokens ({reduction:.1f}%)")

print("=" * 80)


🎬 Review 1:
--------------------------------------------------------------------------------
TRƯỚC (15 từ đầu): ['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'oz', 'episode', 'you']
SAU   (15 từ đầu): ['one', 'reviewers', 'mentioned', 'watching', '1', 'oz', 'episode', "'", 'hooked', '.', 'right', ',', 'exactly', 'happened', '.']
Giảm: 372 → 223 tokens (40.1%)

🎬 Review 2:
--------------------------------------------------------------------------------
TRƯỚC (15 từ đầu): ['a', 'wonderful', 'little', 'production', '.', 'the', 'filming', 'technique', 'is', 'very', 'unassuming', '-', 'very', 'old', '-']
SAU   (15 từ đầu): ['wonderful', 'little', 'production', '.', 'filming', 'technique', 'unassuming', '-', 'old', '-', 'time', '-', 'bbc', 'fashion', 'gives']
Giảm: 195 → 123 tokens (36.9%)


In [ ]:
# Hiển thị sample của subword_tokens để confirm
print("🔍 XEM MẪU SUBWORD TOKENS (dữ liệu chính cho các bước tiếp theo)")
print("=" * 80)

# Lấy 2 reviews đầu tiên
for i in range(2):
    print(f"\n📝 Review {i+1}:")
    print("-" * 80)
    
    # Hiển thị 20 subword tokens đầu tiên
    subword_tokens_sample = df_cleaned.iloc[i]['subword_tokens'][:20]
    
    print(f"Cleaned text: {df_cleaned.iloc[i]['cleaned_review'][:100]}...")
    print(f"\nSubword tokens (20 đầu tiên):")
    print(subword_tokens_sample)
    print(f"\nTổng số subword tokens: {len(df_cleaned.iloc[i]['subword_tokens'])}")

🔍 XEM MẪU SUBWORD TOKENS (dữ liệu chính cho các bước tiếp theo)

📝 Review 1:
--------------------------------------------------------------------------------
Cleaned text: one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. the...

Subword tokens (20 đầu tiên):
['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'oz', 'episode', 'you', "'", 'll', 'be', 'hooked', '.']

Tổng số subword tokens: 372

📝 Review 2:
--------------------------------------------------------------------------------
Cleaned text: a wonderful little production. the filming technique is very unassuming- very old-time-bbc fashion a...

Subword tokens (20 đầu tiên):
['a', 'wonderful', 'little', 'production', '.', 'the', 'filming', 'technique', 'is', 'very', 'unassuming', '-', 'very', 'old', '-', 'time', '-', 'bbc', 'fashion', 'and']

Tổng số subword tokens: 195

✅ Dữ liệu đã sẵn sàng cho các bước tiếp theo!
📌 Sử dụng: df_

## C) Stemming và Lemmatization

#### **Stemming** (Trích xuất gốc từ)
- **Phương pháp**: Cắt bỏ hậu tố (suffix) của từ theo các quy tắc cố định
- **Công cụ**: PorterStemmer (phổ biến nhất)
- **Kết quả**: Có thể ra từ không tồn tại trong từ điển
- **Ví dụ**:
  - "running", "runs", "ran" → "run"
  - "better" → "better" (không đổi)
  - "fishing", "fished", "fisher" → "fish"

#### **Lemmatization** (Chuẩn hóa về dạng gốc)
- **Phương pháp**: Sử dụng từ điển và phân tích từ loại (POS) để tìm dạng gốc
- **Công cụ**: WordNetLemmatizer
- **Kết quả**: Luôn là từ có nghĩa trong từ điển
- **Ví dụ**:
  - "running" (verb) → "run"
  - "better" (adj) → "good" ✓ (cần POS tagging)
  - "was", "were", "is" (verb) → "be"

### Lưu ý với Subword Tokenization

Vì project này sử dụng **Subword Tokenization (BPE)**:
- **Stemming và Lemmatization** thường áp dụng cho **Word Tokenization**
- Với subword tokens, việc stem/lemmatize sẽ **phá vỡ cấu trúc** đã được tối ưu
- **Trong pipeline chính, chúng ta KHÔNG áp dụng** stem/lemmatize cho subword tokens

In [ ]:
resources = ['averaged_perceptron_tagger', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger_eng']
for resource in resources:
    try:
        nltk.data.find(f'taggers/{resource}' if 'tagger' in resource else f'corpora/{resource}')
    except LookupError:
        try:
            nltk.download(resource, quiet=True)
        except:
            pass

# Khởi tạo
porter_stemmer = PorterStemmer()
snowball_stemmer = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()

In [ ]:
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):  # Tính từ (Adjective)
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):  # Động từ (Verb)
        return wordnet.VERB
    elif treebank_tag.startswith('N'):  # Danh từ (Noun)
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):  # Trạng từ (Adverb)
        return wordnet.ADV
    else:
        return wordnet.NOUN  # Mặc định là danh từ

# Test hàm
print(" Test POS tag conversion:")
test_tags = [('better', 'JJR'), ('running', 'VBG'), ('cats', 'NNS')]
for word, tag in test_tags:
    wn_tag = get_wordnet_pos(tag)
    print(f"   '{word}' ({tag}) → WordNet: {wn_tag}")

🔍 Test POS tag conversion:
   'better' (JJR) → WordNet: a
   'running' (VBG) → WordNet: v
   'cats' (NNS) → WordNet: n


### Hàm Stemming và Lemmatization

In [ ]:
def apply_stemming(tokens, stemmer=porter_stemmer):
    return [stemmer.stem(token) for token in tokens]


def apply_lemmatization(tokens):
    # Bước 1: POS tagging - xác định từ loại
    pos_tags = pos_tag(tokens)
    
    # Bước 2: Lemmatize với POS tag tương ứng
    lemmatized = []
    for word, tag in pos_tags:
        wn_tag = get_wordnet_pos(tag)
        lemmatized.append(lemmatizer.lemmatize(word, pos=wn_tag))
    
    return lemmatized

In [ ]:
# Demo với câu review ví dụ
demo_review = "The movies are better and more exciting. The actors were running and fishing beautifully."

print("🎬 DEMO: So sánh Stemming vs Lemmatization")
print("=" * 90)
print(f"Câu gốc:\n{demo_review}\n")

# Tokenize
demo_tokens = word_tokenize(demo_review.lower())
print(f"Tokens gốc ({len(demo_tokens)} từ):")
print(demo_tokens)
print()

# Stemming với PorterStemmer
stemmed_porter = apply_stemming(demo_tokens, porter_stemmer)
print(f"Sau Porter Stemming ({len(stemmed_porter)} từ):")
print(stemmed_porter)
print()

# Lemmatization với POS tagging
lemmatized = apply_lemmatization(demo_tokens)
print(f"Sau Lemmatization + POS ({len(lemmatized)} từ):")
print(lemmatized)
print()

# So sánh chi tiết
print("📊 SO SÁNH CHI TIẾT:")
print("-" * 90)
print(f"{'Từ gốc':<20} {'Porter Stem':<20} {'Lemmatization':<20} {'Ghi chú':<30}")
print("-" * 90)

interesting_words = ['movies', 'are', 'better', 'more', 'exciting', 'actors', 'were', 'running', 'fishing', 'beautifully']
for word in interesting_words:
    if word in demo_tokens:
        idx = demo_tokens.index(word)
        stem = stemmed_porter[idx]
        lem = lemmatized[idx]
        
        note = ""
        if stem != lem:
            note = "⚠️ Khác nhau"
        if lem != word and stem == word:
            note = "✓ Lemma tốt hơn"
        
        print(f"{word:<20} {stem:<20} {lem:<20} {note:<30}")

print("=" * 90)

🎬 DEMO: So sánh Stemming vs Lemmatization
Câu gốc:
The movies are better and more exciting. The actors were running and fishing beautifully.

Tokens gốc (16 từ):
['the', 'movies', 'are', 'better', 'and', 'more', 'exciting', '.', 'the', 'actors', 'were', 'running', 'and', 'fishing', 'beautifully', '.']

Sau Porter Stemming (16 từ):
['the', 'movi', 'are', 'better', 'and', 'more', 'excit', '.', 'the', 'actor', 'were', 'run', 'and', 'fish', 'beauti', '.']

Sau Lemmatization + POS (16 từ):
['the', 'movie', 'be', 'good', 'and', 'more', 'exciting', '.', 'the', 'actor', 'be', 'run', 'and', 'fish', 'beautifully', '.']

📊 SO SÁNH CHI TIẾT:
------------------------------------------------------------------------------------------
Từ gốc               Porter Stem          Lemmatization        Ghi chú                       
------------------------------------------------------------------------------------------
movies               movi                 movie                ⚠️ Khác nhau           

### Áp dụng cho Dataset

In [ ]:
# Áp dụng Stemming cho toàn bộ dataset
df_cleaned['stemmed_tokens'] = df_cleaned['word_tokens'].apply(lambda x: apply_stemming(x, porter_stemmer))

# Áp dụng Lemmatization cho toàn bộ dataset
df_cleaned['lemmatized_tokens'] = df_cleaned['word_tokens'].apply(apply_lemmatization)

🔄 Đang áp dụng Porter Stemming cho dataset...


/var/folders/zc/qmbjdwpx1zj85jxfxr58_vf40000gn/T/ipykernel_73910/721371742.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['stemmed_tokens'] = df_cleaned['word_tokens'].apply(lambda x: apply_stemming(x, porter_stemmer))


✅ Hoàn thành!

🔄 Đang áp dụng Lemmatization cho dataset...
⚠️ Lưu ý: Quá trình này có thể mất vài phút do POS tagging...
✅ Hoàn thành! (áp dụng cho 1000 reviews đầu tiên)


/var/folders/zc/qmbjdwpx1zj85jxfxr58_vf40000gn/T/ipykernel_73910/721371742.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['lemmatized_tokens'] = df_cleaned['word_tokens'].head(1000).apply(apply_lemmatization)


In [ ]:
# So sánh với review thực tế từ dataset
print("📝 SO SÁNH VỚI REVIEW THỰC TẾ TỪ DATASET")
print("=" * 90)

for i in range(2):
    print(f"\n🎬 Review {i+1}:")
    print("-" * 90)
    
    # Lấy 12 từ đầu tiên
    original = df_cleaned.iloc[i]['word_tokens'][:12]
    stemmed = df_cleaned.iloc[i]['stemmed_tokens'][:12]
    lemmatized = df_cleaned.iloc[i]['lemmatized_tokens'][:12] if i < 1000 else ['N/A'] * 12
    
    print(f"Original:    {original}")
    print(f"Stemmed:     {stemmed}")
    print(f"Lemmatized:  {lemmatized}")
    
    # Tính số từ thay đổi
    changed_stem = sum(1 for j in range(len(original)) if original[j] != stemmed[j])
    if i < 1000:
        changed_lem = sum(1 for j in range(len(original)) if original[j] != lemmatized[j])
        print(f"\nSố từ thay đổi: Stemming={changed_stem}/{len(original)}, Lemmatization={changed_lem}/{len(original)}")

print("=" * 90)

📝 SO SÁNH VỚI REVIEW THỰC TẾ TỪ DATASET

🎬 Review 1:
------------------------------------------------------------------------------------------
Original:    ['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1']
Stemmed:     ['one', 'of', 'the', 'other', 'review', 'ha', 'mention', 'that', 'after', 'watch', 'just', '1']
Lemmatized:  ['one', 'of', 'the', 'other', 'reviewer', 'have', 'mention', 'that', 'after', 'watch', 'just', '1']

Số từ thay đổi: Stemming=4/12, Lemmatization=4/12

🎬 Review 2:
------------------------------------------------------------------------------------------
Original:    ['a', 'wonderful', 'little', 'production', '.', 'the', 'filming', 'technique', 'is', 'very', 'unassuming-', 'very']
Stemmed:     ['a', 'wonder', 'littl', 'product', '.', 'the', 'film', 'techniqu', 'is', 'veri', 'unassuming-', 'veri']
Lemmatized:  ['a', 'wonderful', 'little', 'production', '.', 'the', 'filming', 'technique', 'be', 'very', 'unassum